In [1]:
# ============================================================
# Notebook    : 04_aggregate_features.ipynb
# Description : Case A — aggregate (longitudinal) feature
#               engineering, deferred from notebook 02 per
#               design decision (research_plan_v2 §4.1).
#
#               Adds 4 history-summary features, computed with
#               STRICT look-ahead prevention (each row only uses
#               information from years BEFORE that row's Year):
#                 - PrevLabel            : last year's claim flag
#                 - CumClaimCount        : cumulative claims so far
#                 - ClaimRateSoFar       : CumClaimCount / years so far
#                 - YearsSinceFirstClaim : years since first claim
#                                          (large sentinel if none yet)
#
#               NOTE: CumExpoMean deliberately excluded — Expo is
#               already a static feature in ①③, so averaging it
#               over history would conflate "static signal" with
#               "longitudinal signal", defeating the purpose of
#               this comparison.
#
#               Input : data/fremotor_multi_history_features.csv
#                       (notebook 01 output, UNCHANGED)
#               Output: data/fremotor_multi_history_aggregate.csv
# ============================================================


# ============================================================
# 1. Common imports
# ============================================================
import pandas as pd
import numpy as np


# ============================================================
# 2. Load notebook 01 output (unmodified)
# ============================================================
df = pd.read_csv("data/fremotor_multi_history_features.csv")
df = df.sort_values(["IDpol", "Year"]).reset_index(drop=True)

print(f"[CHECK 1] df shape: {df.shape}")
print(f"[CHECK 1] Unique IDpol: {df['IDpol'].nunique():,}")


# ============================================================
# 3. PrevLabel — already exists in notebook 01 output, just
#    confirming it's present and filling first-record NaN with 0
#    (no history yet -> treated as "no prior claim", the neutral
#    default, NOT imputed from any future information)
# ============================================================
assert "PrevLabel" in df.columns, "PrevLabel missing — check notebook 01 output"
df["PrevLabel"] = df["PrevLabel"].fillna(0).astype(int)
print(f"\n[CHECK 2] PrevLabel — NaN count after fill: {df['PrevLabel'].isna().sum()}")


# ============================================================
# 4. CumClaimCount — cumulative claim count STRICTLY BEFORE
#    the current row's year.
#    - groupby().cumsum() on Label would include the CURRENT
#      row's label (look-ahead). We shift(1) first, then cumsum,
#      so row t only sees labels from rows < t within the group.
# ============================================================
df["CumClaimCount"] = (
    df.groupby("IDpol")["Label"]
      .apply(lambda s: s.shift(1).fillna(0).cumsum())
      .reset_index(level=0, drop=True)
).astype(int)

print(f"\n[CHECK 3] CumClaimCount — describe:")
print(df["CumClaimCount"].describe())


# ============================================================
# 5. ClaimRateSoFar — CumClaimCount / years observed so far
#    (years so far = YearGap, since YearGap=0 at first record
#    means 0 prior years observed)
#    - YearGap=0 rows have no prior years -> rate defined as 0
#      (not NaN), consistent with "no evidence of risk yet"
# ============================================================
df["ClaimRateSoFar"] = np.where(
    df["YearGap"] > 0,
    df["CumClaimCount"] / df["YearGap"],
    0.0
)

print(f"\n[CHECK 4] ClaimRateSoFar — describe:")
print(df["ClaimRateSoFar"].describe())


# ============================================================
# 6. YearsSinceFirstClaim — years elapsed since the policyholder's
#    FIRST claim, using only claims STRICTLY BEFORE the current
#    row (same shift(1) discipline as CumClaimCount).
#    - Uses PrevLabel-based cumulative claim flag internally to
#      find "has there been any claim before this row" and "which
#      year was the first one", all from prior rows only.
#    - Sentinel value: if no prior claim exists yet, set to a
#      large constant (999) rather than NaN, matching the
#      convention that "no signal yet" should be an explicit,
#      learnable value rather than a missing one (LightGBM/
#      Transformer would otherwise need separate NaN-handling
#      logic that the other three features don't require).
# ============================================================
NO_PRIOR_CLAIM_SENTINEL = 999

def compute_years_since_first_claim(group):
    group = group.sort_values("Year")
    prior_claim_years = group.loc[group["PrevLabel"] == 1, "Year"] - 1
    # ^ PrevLabel==1 at row t means a claim happened in year (t-1);
    #   we want the YEAR that claim occurred in, i.e. group["Year"] - 1
    #   for rows where PrevLabel==1. This is still "prior information
    #   as seen at row t" — no use of row t's own Label.

    first_claim_year = prior_claim_years.min() if len(prior_claim_years) > 0 else None

    if first_claim_year is None:
        result = pd.Series(NO_PRIOR_CLAIM_SENTINEL, index=group.index)
    else:
        result = (group["Year"] - first_claim_year).clip(lower=0)
        # rows before the first claim year still get sentinel
        result = np.where(group["Year"] > first_claim_year,
                           result, NO_PRIOR_CLAIM_SENTINEL)
        result = pd.Series(result, index=group.index)
    return result

df["YearsSinceFirstClaim"] = (
    df.groupby("IDpol", group_keys=False).apply(compute_years_since_first_claim)
).astype(int)

print(f"\n[CHECK 5] YearsSinceFirstClaim — describe:")
print(df["YearsSinceFirstClaim"].describe())
print(f"[CHECK 5] Rows still at sentinel (999, no prior claim): "
      f"{(df['YearsSinceFirstClaim'] == NO_PRIOR_CLAIM_SENTINEL).sum():,} "
      f"({(df['YearsSinceFirstClaim'] == NO_PRIOR_CLAIM_SENTINEL).mean()*100:.1f}%)")


# ============================================================
# 7. Look-ahead sanity check — spot-check a few multi-claim
#    policyholders by hand to confirm no future information
#    leaked into any of the four new columns
# ============================================================
sample_idpol = df[df["CumClaimCount"] > 0]["IDpol"].iloc[0]
print(f"\n[CHECK 6] Manual look-ahead check for IDpol={sample_idpol}:")
print(df[df["IDpol"] == sample_idpol][
    ["IDpol", "Year", "YearGap", "Label", "PrevLabel",
     "CumClaimCount", "ClaimRateSoFar", "YearsSinceFirstClaim"]
].to_string(index=False))


# ============================================================
# 8. Save
# ============================================================
df.to_csv("data/fremotor_multi_history_aggregate.csv", index=False)
print(f"\nSaved: data/fremotor_multi_history_aggregate.csv")


# ============================================================
# 9. Summary check
# ============================================================
print("\n===== Aggregate Feature Engineering Summary =====")
print(f"New columns added : PrevLabel(reused), CumClaimCount, "
      f"ClaimRateSoFar, YearsSinceFirstClaim")
print(f"CumExpoMean        : deliberately excluded (see notebook header)")
print(f"Look-ahead policy  : all features use shift(1)/strictly-prior logic")
print(f"Output shape       : {df.shape}")
print(f"Output file        : data/fremotor_multi_history_aggregate.csv")
print("===================================================")

[CHECK 1] df shape: (364997, 11)
[CHECK 1] Unique IDpol: 71,392

[CHECK 2] PrevLabel — NaN count after fill: 0

[CHECK 3] CumClaimCount — describe:
count    364997.000000
mean          0.330055
std           0.766222
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           8.000000
Name: CumClaimCount, dtype: float64

[CHECK 4] ClaimRateSoFar — describe:
count    364997.000000
mean          0.101744
std           0.233540
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
Name: ClaimRateSoFar, dtype: float64


C:\Users\User\AppData\Local\Temp\ipykernel_16004\271147661.py:127: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("IDpol", group_keys=False).apply(compute_years_since_first_claim)



[CHECK 5] YearsSinceFirstClaim — describe:
count    364997.000000
mean        787.776384
std         407.173843
min           1.000000
25%         999.000000
50%         999.000000
75%         999.000000
max         999.000000
Name: YearsSinceFirstClaim, dtype: float64
[CHECK 5] Rows still at sentinel (999, no prior claim): 287,601 (78.8%)

[CHECK 6] Manual look-ahead check for IDpol=PN100029:
   IDpol  Year  YearGap  Label  PrevLabel  CumClaimCount  ClaimRateSoFar  YearsSinceFirstClaim
PN100029  2003        0      0          0              0        0.000000                   999
PN100029  2004        1      1          0              0        0.000000                   999
PN100029  2005        2      1          1              1        0.500000                     1
PN100029  2006        3      0          1              2        0.666667                     2
PN100029  2007        4      0          0              2        0.500000                     3

Saved: data/fremotor_multi_hist

In [2]:
# ============================================================
# 6-B. Fix YearsSinceFirstClaim scale distortion
#      (patch — run this AFTER Section 6, before Section 7)
#
#      Problem: the 999 sentinel dominates 78.8% of rows, so any
#      scaler (MinMax etc.) downstream would compress the real
#      1-9 year range into a tiny sliver near 0.
#
#      Fix: split into two features —
#        - HasPriorClaim         : 1 if any prior claim exists, else 0
#        - YearsSinceFirstClaimLog : log1p(years) if HasPriorClaim,
#                                     else 0 (safe because
#                                     HasPriorClaim=0 already flags
#                                     this 0 as "not a real year count")
#      The raw YearsSinceFirstClaim (with 999 sentinel) is kept in
#      the dataframe for reference but should NOT be fed to 03c/04b
#      directly — use the two derived columns instead.
# ============================================================
df["HasPriorClaim"] = (df["YearsSinceFirstClaim"] != NO_PRIOR_CLAIM_SENTINEL).astype(int)

df["YearsSinceFirstClaimLog"] = np.where(
    df["HasPriorClaim"] == 1,
    np.log1p(df["YearsSinceFirstClaim"]),
    0.0
)

print(f"\n[CHECK 5-B] HasPriorClaim distribution:")
print(df["HasPriorClaim"].value_counts())
print(f"\n[CHECK 5-B] YearsSinceFirstClaimLog — describe (HasPriorClaim==1 only):")
print(df.loc[df["HasPriorClaim"] == 1, "YearsSinceFirstClaimLog"].describe())

# re-run the same manual check to confirm the split is consistent
print(f"\n[CHECK 5-B] Same IDpol as CHECK 6, with new columns:")
print(df[df["IDpol"] == sample_idpol][
    ["IDpol", "Year", "Label", "PrevLabel", "CumClaimCount",
     "YearsSinceFirstClaim", "HasPriorClaim", "YearsSinceFirstClaimLog"]
].to_string(index=False))


[CHECK 5-B] HasPriorClaim distribution:
HasPriorClaim
0    287601
1     77396
Name: count, dtype: int64

[CHECK 5-B] YearsSinceFirstClaimLog — describe (HasPriorClaim==1 only):
count    77396.000000
mean         1.255331
std          0.444708
min          0.693147
25%          0.693147
50%          1.098612
75%          1.609438
max          2.197225
Name: YearsSinceFirstClaimLog, dtype: float64

[CHECK 5-B] Same IDpol as CHECK 6, with new columns:
   IDpol  Year  Label  PrevLabel  CumClaimCount  YearsSinceFirstClaim  HasPriorClaim  YearsSinceFirstClaimLog
PN100029  2003      0          0              0                   999              0                 0.000000
PN100029  2004      1          0              0                   999              0                 0.000000
PN100029  2005      1          1              1                     1              1                 0.693147
PN100029  2006      0          1              2                     2              1                 1.098

In [3]:
df.to_csv("data/fremotor_multi_history_aggregate.csv", index=False)
print(f"\nSaved: data/fremotor_multi_history_aggregate.csv")

print("\n===== Aggregate Feature Engineering Summary =====")
print(f"New columns added : PrevLabel(reused), CumClaimCount, ClaimRateSoFar,")
print(f"                    HasPriorClaim, YearsSinceFirstClaimLog")
print(f"                    (raw YearsSinceFirstClaim kept for reference only)")
print(f"CumExpoMean        : deliberately excluded (see notebook header)")
print(f"Look-ahead policy  : all features use shift(1)/strictly-prior logic")
print(f"Output shape       : {df.shape}")
print(f"Output file        : data/fremotor_multi_history_aggregate.csv")
print("===================================================")


Saved: data/fremotor_multi_history_aggregate.csv

===== Aggregate Feature Engineering Summary =====
New columns added : PrevLabel(reused), CumClaimCount, ClaimRateSoFar,
                    HasPriorClaim, YearsSinceFirstClaimLog
                    (raw YearsSinceFirstClaim kept for reference only)
CumExpoMean        : deliberately excluded (see notebook header)
Look-ahead policy  : all features use shift(1)/strictly-prior logic
Output shape       : (364997, 16)
Output file        : data/fremotor_multi_history_aggregate.csv
